# Composer classification — first pass

Goal: predict the composer of a piece from its MIDI file (9 classes). Two models are trained on a shared
preprocessing pipeline, per the project instructions: a CNN and an LSTM.

Pipeline: MIDI file → binary piano roll (8 frames/second × 128 pitches) → fixed-length windows
(240 frames = 30 s, 50% overlap) → window-level training → piece-level evaluation by averaging
window probabilities.

Environment: conda env `511-team-project` (TensorFlow, pretty_midi, scikit-learn). EDA is deferred
to a later pass.

In [1]:
import os
import glob

import numpy as np
import pretty_midi
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = "../build/datasets/Composer_Dataset/NN_midi_files_extended"
CACHE_DIR = "cache"
FS = 8        # piano-roll frames per second
WINDOW = 240  # frames per example (30 s)
HOP = 120     # 50% overlap between windows

COMPOSERS = sorted(d for d in os.listdir(os.path.join(DATA_DIR, "train")) if not d.startswith("."))
print(len(COMPOSERS), "composers:", COMPOSERS)

9 composers: ['bach', 'bartok', 'byrd', 'chopin', 'handel', 'hummel', 'mendelssohn', 'mozart', 'schumann']


## Building the dataset from individual MIDI files

MIDI is symbolic, not audio: each file is a list of timed note events (note-on/note-off, pitch 0–127,
velocity). `pretty_midi` reads those events and renders a **piano roll** — time discretized into
`FS` frames per second, each frame a 128-dimensional vector of sounding pitches, which we binarize.

A whole piece is one labeled example but pieces vary in length, so each piano roll is cut into
overlapping fixed-length windows. Every window inherits its piece's composer label, and the piece id
is kept so window predictions can be aggregated back to piece level at evaluation time.

In [2]:
def midi_to_piano_roll(path, fs=FS):
    """Parse one MIDI file into a binary piano roll of shape (time, 128)."""
    pm = pretty_midi.PrettyMIDI(path)
    roll = pm.get_piano_roll(fs=fs)          # (128, T) velocity floats
    return (roll > 0).astype(np.uint8).T     # (T, 128) binary


def windows_from_roll(roll, window=WINDOW, hop=HOP):
    """Cut a piano roll into fixed-length windows; zero-pad pieces shorter than one window."""
    if len(roll) < window:
        padded = np.zeros((window, roll.shape[1]), dtype=roll.dtype)
        padded[: len(roll)] = roll
        return [padded]
    return [roll[start : start + window] for start in range(0, len(roll) - window + 1, hop)]


def build_split(split):
    X, y, piece_ids, skipped = [], [], [], []
    for label, composer in enumerate(COMPOSERS):
        for path in sorted(glob.glob(os.path.join(DATA_DIR, split, composer, "*.mid"))):
            try:
                roll = midi_to_piano_roll(path)
            except Exception as e:
                skipped.append((os.path.basename(path), str(e)[:60]))
                continue
            for w in windows_from_roll(roll):
                X.append(w)
                y.append(label)
                piece_ids.append(os.path.basename(path))
    if skipped:
        print(f"{split}: skipped {len(skipped)} unreadable files, e.g. {skipped[:3]}")
    return np.stack(X), np.array(y), np.array(piece_ids)

In [3]:
# Parse once, cache to disk
os.makedirs(CACHE_DIR, exist_ok=True)
data = {}
for split in ("train", "dev", "test"):
    cache = os.path.join(CACHE_DIR, f"{split}_fs{FS}_w{WINDOW}_h{HOP}.npz")
    if os.path.exists(cache):
        z = np.load(cache)
        data[split] = (z["X"], z["y"], z["ids"])
    else:
        X, y, ids = build_split(split)
        np.savez_compressed(cache, X=X, y=y, ids=ids)
        data[split] = (X, y, ids)
    X, y, ids = data[split]
    print(f"{split}: {X.shape[0]} windows of shape {X.shape[1:]} from {len(np.unique(ids))} pieces")

/opt/anaconda3/envs/511-team-project/lib/python3.11/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


train: 7371 windows of shape (240, 128) from 369 pieces
dev: 672 windows of shape (240, 128) from 35 pieces
test: 742 windows of shape (240, 128) from 35 pieces


## CNN

The window is treated as a 240 × 128 single-channel image: composer style shows up as visual texture
(chord shapes, voice spacing, rhythmic density).

In [4]:
def make_cnn():
    return keras.Sequential(
        [
            layers.Input(shape=(WINDOW, 128, 1)),
            layers.Conv2D(32, 3, padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2),
            layers.Conv2D(64, 3, padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2),
            layers.Conv2D(128, 3, padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2),
            layers.GlobalAveragePooling2D(),
            layers.Dropout(0.3),
            layers.Dense(len(COMPOSERS), activation="softmax"),
        ],
        name="cnn",
    )


make_cnn().summary()

Model: "cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 240, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 240, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 120, 64, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 120, 64, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 120, 64, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 60, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 9)              │         1,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,729 (370.04 KB)

 Trainable params: 94,281 (368.29 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
Xtr, ytr, _ = data["train"]
Xdev, ydev, _ = data["dev"]


def to_cnn(X):
    return X[..., None].astype(np.float32)


cnn = make_cnn()
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_cnn = cnn.fit(
    to_cnn(Xtr),
    ytr,
    validation_data=(to_cnn(Xdev), ydev),
    epochs=50,
    batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=2,
)

Epoch 1/50


## LSTM

The same window viewed as a sequence: 240 time steps, each a 128-dimensional frame vector.
Training is slower than the CNN because recurrence cannot be parallelized across time.

In [ ]:
def make_lstm():
    return keras.Sequential(
        [
            layers.Input(shape=(WINDOW, 128)),
            layers.LSTM(128, return_sequences=True),
            layers.LSTM(128),
            layers.Dropout(0.3),
            layers.Dense(len(COMPOSERS), activation="softmax"),
        ],
        name="lstm",
    )


lstm = make_lstm()
lstm.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_lstm = lstm.fit(
    Xtr.astype(np.float32),
    ytr,
    validation_data=(Xdev.astype(np.float32), ydev),
    epochs=50,
    batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=2,
)

## Evaluation

Window-level metrics measure the classifier; piece-level metrics (averaging window probabilities per
piece) measure the actual use case: naming the composer of a whole score.

In [ ]:
def evaluate(model, split, is_cnn):
    X, y, ids = data[split]
    Xf = to_cnn(X) if is_cnn else X.astype(np.float32)
    probs = model.predict(Xf, batch_size=64, verbose=0)

    print(f"=== {model.name} / {split} — window level")
    print(classification_report(y, probs.argmax(1), target_names=COMPOSERS,
                                labels=range(len(COMPOSERS)), digits=3, zero_division=0))

    piece_true, piece_pred = [], []
    for pid in np.unique(ids):
        m = ids == pid
        piece_true.append(int(y[m][0]))
        piece_pred.append(int(probs[m].mean(axis=0).argmax()))
    print(f"=== {model.name} / {split} — piece level")
    print(classification_report(piece_true, piece_pred, target_names=COMPOSERS,
                                labels=range(len(COMPOSERS)), digits=3, zero_division=0))
    return np.array(piece_true), np.array(piece_pred)


for model, is_cnn in ((cnn, True), (lstm, False)):
    evaluate(model, "dev", is_cnn)

In [ ]:
# Final numbers on the held-out test split
results = {}
for model, is_cnn in ((cnn, True), (lstm, False)):
    results[model.name] = evaluate(model, "test", is_cnn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (name, (true, pred)) in zip(axes, results.items()):
    cm = confusion_matrix(true, pred, labels=range(len(COMPOSERS)))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(COMPOSERS)), COMPOSERS, rotation=45, ha="right")
    ax.set_yticks(range(len(COMPOSERS)), COMPOSERS)
    ax.set_title(f"{name} — piece-level confusion (test)")
    for i in range(len(COMPOSERS)):
        for j in range(len(COMPOSERS)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
axes[0].set_ylabel("true composer")
fig.supxlabel("predicted composer")
plt.tight_layout()
plt.show()

## Next steps

- EDA pass: piece lengths, polyphony density, pitch-range profiles per composer.
- Hyperparameter tuning: window length/overlap, frame rate, units/filters, dropout, learning rate.
- Report material: training curves, per-composer error analysis from the confusion matrices.